# Sahayak - ORBIT pipeline on Colab (resume-safe: everything is saved to Google Drive after each step)
Setup once: **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**. Keep this tab open and the laptop awake (about 3 hours).
If Colab disconnects, reconnect and **Run all** again: finished steps are skipped (their results are in Drive `MyDrive/sahayak_orbit_out`).
Order: data -> main model C -> cross-video model -> memory diagnosis (val split) -> LoRA -> baselines -> plot.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, zipfile, re, shutil, requests, glob, json
OUT = '/content/drive/MyDrive/sahayak_orbit_out'; os.makedirs(OUT, exist_ok=True)
os.chdir('/content')
!rm -rf sahayak2
!git clone -q https://github.com/shreyaaaah/fewshot-object-detection-for-visually-impaired-person.git sahayak2
os.chdir('/content/sahayak2')
!pip -q install timm scikit-learn tqdm pyyaml
!nvidia-smi | head -12
os.makedirs('reports', exist_ok=True); os.makedirs('checkpoints', exist_ok=True)
for d in ('reports', 'checkpoints'):                      # restore earlier results after a reconnect
    if os.path.isdir(f'{OUT}/{d}'):
        shutil.copytree(f'{OUT}/{d}', d, dirs_exist_ok=True)
RAW, ROOT = '/content/orbit_raw', '/content/orbit_instances'
EPS = 1500
EV = f'--root {ROOT} --split test --episodes 200 --k-shots 1 3 5 10 --cross-video'

def save():
    for d in ('reports', 'checkpoints'):
        shutil.copytree(d, f'{OUT}/{d}', dirs_exist_ok=True)

def sh(cmd):
    get_ipython().system(cmd)

def finished(log):
    p = f'{OUT}/reports/{log}'
    return os.path.exists(p) and 'done. best val score' in open(p).read()

def train(tag, extra):
    if finished(f'log_train_{tag}.txt'):
        print('skip training', tag); return
    sh(f'python -m src.train.train --root {ROOT} --loss open {extra} --episodes {EPS} --tag {tag} --amp 2>&1 | tee reports/log_train_{tag}.txt | tail -5')
    save()

def evaluate(tag, extra=''):
    if os.path.exists(f'{OUT}/reports/episodic_{tag}_test.json'):
        print('skip eval', tag); return
    sh(f'python -m src.eval.episodic {EV} {extra} --tag {tag} 2>&1 | tee reports/log_eval_{tag}.txt | tail -45')
    save()

def diag(tag, extra):
    if os.path.exists(f'{OUT}/reports/log_diag_{tag}.txt'):
        print('skip diag', tag); return
    sh(f'python tools/diagnose_memory.py --root {ROOT} --split val {extra} --episodes 30 2>&1 | tee reports/log_diag_{tag}.txt')
    save()


In [ ]:
# DATA: from Drive if saved earlier, else download ORBIT (about 40 min) and save a copy in Drive
if os.path.exists(f'{OUT}/orbit_instances.tar') and not os.path.isdir(ROOT):
    sh(f'tar -xf {OUT}/orbit_instances.tar -C /content')
if not os.path.isdir(ROOT) or len(os.listdir(ROOT)) < 100:
    files = requests.get('https://api.figshare.com/v2/articles/14294597', timeout=60).json()['files']
    byname = {f['name']: f for f in files}
    os.makedirs(RAW, exist_ok=True)
    EVERY = 10
    def fetch(name):
        f = byname[name]; z = '/content/' + name
        print('downloading', name, round(f['size'] / 1e9, 1), 'GB ...', flush=True)
        subprocess.run(['wget', '-q', '-c', '-O', z, f['download_url']], check=True)
        kept = total = 0
        pat = re.compile(r'(\d+)\.(?:jpe?g|png)$', re.I)
        with zipfile.ZipFile(z) as zf:
            for m in zf.infolist():
                if m.is_dir():
                    continue
                total += 1
                k = pat.search(m.filename)
                if k and int(k.group(1)) % EVERY != 0:
                    continue
                zf.extract(m, RAW); kept += 1
        os.remove(z)
        print(f'{name}: kept {kept} of {total} frames', flush=True)
    fetch('validation_224.zip'); fetch('train_224.zip')
    sh(f'python tools/prepare_orbit.py --src {RAW} --out {ROOT} --every 1 2>&1 | tail -5')
    shutil.rmtree(RAW, ignore_errors=True)
    sh(f'tar -cf /content/orbit_instances.tar -C /content orbit_instances && cp /content/orbit_instances.tar {OUT}/ && rm /content/orbit_instances.tar')
n = len(os.listdir(ROOT)) if os.path.isdir(ROOT) else 0
print(n, 'object folders in ROOT (expect about 330)')
assert n > 100, 'conversion found too little data - paste the output above to Claude'


## A. Main model C (last 2 blocks, open-set loss) + test evaluation

In [ ]:
train('C_partial_open', '--mode partial')
evaluate('C_partial_open', '--ckpt checkpoints/embedder_C_partial_open.pt')


## B. Cross-video model (episodes follow the test protocol: support from clean, queries from clutter)

In [ ]:
train('C_partial_open_xv', '--mode partial --cross-video')
evaluate('C_partial_open_xv', '--ckpt checkpoints/embedder_C_partial_open_xv.pt')


## C. Memory diagnosis on the VALIDATION split only (read-only)

In [ ]:
diag('B_frozen', '--frozen-backbone vit_small_patch14_dinov2.lvd142m')
diag('C', '--ckpt checkpoints/embedder_C_partial_open.pt')
diag('C_xv', '--ckpt checkpoints/embedder_C_partial_open_xv.pt')


## D. LoRA variant

In [ ]:
train('D_lora_open', '--mode lora')
evaluate('D_lora_open', '--ckpt checkpoints/embedder_D_lora_open.pt')


## E. Baselines (for the comparison plot)

In [ ]:
evaluate('A0_hsv_hist', '--frozen-backbone hsv_hist')
evaluate('A_resnet50_frozen', '--frozen-backbone resnet50.a1_in1k')
evaluate('B_dinov2_frozen')


## F. Plot and table

In [ ]:
import matplotlib.pyplot as plt, pathlib
R = {}
for f in sorted(glob.glob('reports/episodic_*_test.json')):
    R[pathlib.Path(f).stem.replace('episodic_', '').replace('_test', '')] = json.load(open(f))['results']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, res in R.items():
    ks = sorted(int(k) for k in res)
    for a, key, scale, ttl in ((ax[0], 'closed_acc', 100, 'Closed-set accuracy (%) vs shots K'),
                               (ax[1], 'auroc_ours_prob', 1, 'Open-set AUROC vs K')):
        a.errorbar(ks, [res[str(k)][key][0] * scale for k in ks], yerr=[res[str(k)][key][1] * scale for k in ks],
                   marker='o', capsize=3, label=name)
        a.set_title(ttl); a.set_xlabel('K (examples per object)'); a.grid(alpha=.3)
ax[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig('reports/orbit_results.png', dpi=150); plt.show()
save(); print('Saved to Drive:', OUT); print(sorted(os.listdir(f'{OUT}/reports'))); print(sorted(os.listdir(f'{OUT}/checkpoints')))
